In [3]:
import sys
sys.path.append('../../Simulate/')

import os
import random
import numpy as np
import subprocess

from Bio import SeqIO
from tqdm import tqdm
from scipy.stats import bernoulli
from typing import Dict, Union, Tuple
from threading import Lock
from concurrent.futures import ThreadPoolExecutor

from LockedIterator_queue import LockedIterator
from SetMethylation import SetMethylation
# from StreamReads import StreamReads
# from StreamHTSIM import StreamHTSIM
from UtilityFunctions import get_htsim_path
from ParseGenome import ParseGenome

ModuleNotFoundError: No module named 'ParseGenome'

In [5]:
import importlib
import BSReadSim_queue

importlib.reload(BSReadSim_queue)
from BSReadSim_queue import BSReadSim

In [3]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"

ref_fasta = working_path + "data/ref/BSB_test.fa"
outdir = working_path + "outdir"

cgmap_file  = working_path + "data/sim/pe_d/sim.CGmap.gz"
asm_file    = working_path + "data/sim/pe_d/sim.asm.gz"

# sequential test

In [4]:
self = BSReadSim(ref_fasta=ref_fasta, outdir=outdir, 
                 cgmap_file=cgmap_file, asm_file = asm_file,
                 n_threads=1, num_reads=1000, 
                 overwrite_db=True, verbose =True, shuffle=False, gzip=False)

Initiating experiment...
Initiating methylation profile...

[Initiating meth_db] for chr10...
Filling CGmap for chr10... 184989 sites found in CGmap file, among them 0 sites (0.0%) are incompatible...
Filling ASM for chr10... 12680 sites found in CGmap file, among them 0 sites (0.0%) are incompatible...
Filling with beta distribution for chr10...
Processed 187408 sites from contig chr10

[Initiating meth_db] for chr11...
Filling CGmap for chr11... 187526 sites found in CGmap file, among them 0 sites (0.0%) are incompatible...
Filling ASM for chr11... 12947 sites found in CGmap file, among them 0 sites (0.0%) are incompatible...
Filling with beta distribution for chr11...
Processed 187844 sites from contig chr11

[Initiating meth_db] for chr12...
Filling CGmap for chr12... 183947 sites found in CGmap file, among them 0 sites (0.0%) are incompatible...
Filling ASM for chr12... 12318 sites found in CGmap file, among them 0 sites (0.0%) are incompatible...
Filling with beta distribution fo

../../Simulate/StreamReads.py:42: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir/sim_1.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')
../../Simulate/StreamReads.py:42: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir/sim_2.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')


In [ ]:
for contig_id in self.count_dict.keys():
    sim_cmd  = self.cmd_part + ['-c', contig_id] + ['-n', str(self.count_dict[contig_id])]
    read_gen = LockedIterator(StreamHTSIM(sim_cmd=sim_cmd, pair_end=self.pair_end)) # only output 1 header for -c TODO
    var_contig, sim_data= next(read_gen)                                    # the first element of generator is variants
    self.current_contig = var_contig                                        # update the profiles
    self.pos_map, self.meth_arr, _ = self.meth_db.load_contig(var_contig)   # [pos_map, meth_arr, status]
    self.variant_profile= self.meth_set.set_var_meth(var_contig, sim_data)  # a dict, can be empty
    
    for _, read_pair in read_gen:
        read1_idx   = random.choice([0, 1]) 
        pattern_idx = random.choice([0, 1]) if self.undirectional else read1_idx
        strand_idx  = random.choice([0, 1]) if read_pair[0]['strand']<0 else read_pair[0]['strand']
        read_pair[1-read1_idx]['read2'] = 1
        read_pair[0]['conv'] = pattern_idx
        read_pair[1]['conv'] = pattern_idx
        read_pair[0]['strand'] = strand_idx
        read_pair[1]['strand'] = strand_idx
        
        # mask the context
        self.mask_context(read_pair[0])
        self.mask_context(read_pair[1])
        
        # retrive methy profile
        self.retrive_meth_db(read_pair[0])
        self.retrive_meth_db(read_pair[1])

        # set methylation states
        self.set_context_state(read_pair)

        # bisulfite converted
        self.treat_bisulfite(read_pair[0])
        self.treat_bisulfite(read_pair[1])

        # rev complementary
        self.rev_complement(read_pair)

        # introduce seq errors
        self.add_seq_err(read_pair[0])
        self.add_seq_err(read_pair[1])
        
        # introduce quality scores
        self.add_qual_score(read_pair[0])
        self.add_qual_score(read_pair[1])
        
#         # output
#         self.fastq_out.output_reads(read_pair)

# self.fastq_out.close()

In [ ]:
self.pos_map, self.meth_arr, _ = self.meth_db.load_contig("chr10")  

for row in self.meth_arr:
    if np.isnan(row[4]):
        print(row)
        break

In [ ]:
np.where(self.meth_arr[:, 3] !=self.meth_arr[:, 4])[0].size

In [ ]:
read_pair

In [ ]:
from pympler import asizeof
print(asizeof.asizeof(read_pair))

# multiprocess

In [4]:
self = BSReadSim(ref_fasta=ref_fasta, outdir=outdir, 
                 meth_db_path=outdir,
                 n_threads=16, num_reads=100000, 
                 overwrite_db=True, verbose =True, shuffle=False, gzip=False)

Initiating genome...
Initiating methylation profile...

[Initiating meth_db] for chr10...

[Initiating meth_db] for chr11...

[Initiating meth_db] for chr12...

[Initiating meth_db] for chr13...

[Initiating meth_db] for chr14...

[Initiating meth_db] for chr15...


../../Simulate/StreamReads_queue.py:39: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir/sim_1.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')
../../Simulate/StreamReads_queue.py:39: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir/sim_2.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')


In [5]:
self.run()

Simulating methylated Reads with 16 threads...
[CMD]: /home/wbguo/iproject/BSReadSim/HTSIM/htsim /home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa -i 400 -I 25 -m 100 -M 1000 -1 100 -2 100 -e 0 -A 0.05 -u 1 -f 1 -g None -r 0.001 -R 0.15 -X 0.15 -h 0 -s -1 -T 0 -x None -b None -B None -D None
[INFO]: #reads/#read pairs for each contig {'chr10': 10794, 'chr11': 10808, 'chr12': 10749, 'chr13': 8569, 'chr14': 8953, 'chr15': 127}


  0%|                                                                                                                                                                                    | 0/50000 [00:00<?, ?it/s]Simulating whole genome reads:
Reference genome file: /home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa
[main] Calculating the total length and effective length of the reference sequences...
[main] Contig chr10 specified, contig length: 423500, effective length: 423500
[main] No VCF input, will generate SNP randomly if mutation rate is nonzero
[htsim] seed = 1678734920
[sim_core] contig 'chr10': simulate 10794 reads...
Exception in thread Thread-11:
Traceback (most recent call last):
  File "/home/wbguo/apps/anaconda3/lib/python3.8/threading.py", line 932, in _bootstrap_inner
    self.run()
  File "/home/wbguo/apps/anaconda3/lib/python3.8/threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multipro

  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/queues.py", line 355, in get
    with self._rlock:
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/queues.py", line 355, in get
    with self._rlock:
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/queues.py", line 355, in get
    with self._rlock:
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/queues.py", line 355, in get
    with self._rlock:
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/queues.py", line 355, in get
    with self._rlock:
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/queues.py", line 355, in get
    with self._rlock:
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/queues.py", lin

KeyboardInterrupt: 

Process ForkPoolWorker-24:
Process ForkPoolWorker-30:
Process ForkPoolWorker-27:
Process ForkPoolWorker-26:
Process ForkPoolWorker-32:
Process ForkPoolWorker-28:
Process ForkPoolWorker-22:
Process ForkPoolWorker-17:
Process ForkPoolWorker-18:
Process ForkPoolWorker-29:
Process ForkPoolWorker-20:
Process ForkPoolWorker-25:
Process ForkPoolWorker-19:
Process ForkPoolWorker-31:
Process ForkPoolWorker-23:
Process ForkPoolWorker-21:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/process.py", 

  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/queues.py", line 355, in get
    with self._rlock:
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/queues.py", line 355, in get
    with self._rlock:
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/queues.py", line 355, in get
    with self._rlock:
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/queues.py", line 355, i

In [6]:
import threading
import multiprocessing
from StreamHTSIM import StreamHTSIM

print(f'Simulating methylated Reads with {self.n_threads} threads...')
print(f'[CMD]: {" ".join(self.cmd_part)}')
if self.verbose:
    print(f'[INFO]: #reads/#read pairs for each contig', end=" ")
    print(self.count_dict)


Simulating methylated Reads with 16 threads...
[CMD]: /home/wbguo/iproject/BSReadSim/HTSIM/htsim /home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa -i 400 -I 25 -m 100 -M 1000 -1 100 -2 100 -e 0 -A 0.05 -u 1 -f 1 -g None -r 0.001 -R 0.15 -X 0.15 -h 0 -s -1 -T 0 -x None -b None -B None -D None
[INFO]: #reads/#read pairs for each contig {'chr10': 10794, 'chr11': 10808, 'chr12': 10749, 'chr13': 8569, 'chr14': 8953, 'chr15': 127}


In [17]:
contig_id = 'chr10'

In [18]:
sim_cmd  = self.cmd_part + ['-c', contig_id] + ['-n', str(self.count_dict[contig_id])]
read_gen = LockedIterator(StreamHTSIM(sim_cmd=sim_cmd, pair_end=self.pair_end)) # only output 1 header for -c TODO:
var_contig, sim_data= next(read_gen)                                    # the first element of generator is variants
self.current_contig = var_contig                                        # update the profiles
self.pos_map, self.meth_arr, _ = self.meth_db.load_contig(var_contig)   # [pos_map, meth_arr, status]
self.variant_profile= self.meth_set.set_var_meth(var_contig, sim_data)  # a dict, can be empty

Simulating whole genome reads:
Reference genome file: /home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa
[main] Calculating the total length and effective length of the reference sequences...
[main] Contig chr10 specified, contig length: 423500, effective length: 423500
[main] No VCF input, will generate SNP randomly if mutation rate is nonzero
[htsim] seed = 1678735183
[sim_core] contig 'chr10': simulate 10794 reads...


In [19]:
pool = multiprocessing.Pool(processes=self.n_threads)

[sim_core] Generated 10794 read pairs, with 1198 contain SNP, 284 contain INDEL


In [15]:
def process_read_pair(read_pair):
    """
    This processing step works as follows:
    1. check if strand capture (targeted sequencing), if yes assign strandness accordingly
    2. if not, randomly assign reads to Watson or Crick strand, with corresponding base change pattern
    3. set methylation status according to methylation profile
    4. bisulfite converted and introduce sequencing error
    5. output reads
    """

    # for directional library, read1 always C2T, strand can be either watson or crick unless strand captured
    read1_idx   = random.choice([0, 1]) 
    pattern_idx = random.choice([0, 1]) if self.undirectional else read1_idx
    strand_idx  = random.choice([0, 1]) if read_pair[0]['strand']<0 else read_pair[0]['strand']
    read_pair[1-read1_idx]['read2'] = 1
    read_pair[0]['conv'] = pattern_idx
    read_pair[1]['conv'] = pattern_idx
    read_pair[0]['strand'] = strand_idx
    read_pair[1]['strand'] = strand_idx

    # mask the context
    self.mask_context(read_pair[0])
    self.mask_context(read_pair[1])

    # retrive methy profile
    self.retrive_meth_db(read_pair[0])
    self.retrive_meth_db(read_pair[1])

    # set methylation states
    self.set_context_state(read_pair)

    # bisulfite converted
    self.treat_bisulfite(read_pair[0])
    self.treat_bisulfite(read_pair[1])

    # rev complementary
    self.rev_complement(read_pair)

    # introduce seq errors
    self.add_seq_err(read_pair[0])
    self.add_seq_err(read_pair[1])

    # introduce quality scores
    self.add_qual_score(read_pair[0])
    self.add_qual_score(read_pair[1])

    self.data_queue.put(read_pair)
    print(self.data_queue.qsize())

In [20]:
pool = multiprocessing.Pool(processes=self.n_threads)
results = []
for _, read_pair in read_gen:
    result = pool.apply_async(self.process_read_pair, args=(read_pair,), 
                              callback=self.update_progress, error_callback=self.process_error)
    results.append(result)

 # Wait for all processes to complete
for result in results:
    result.wait()

Exception in thread Thread-31:
Traceback (most recent call last):
  File "/home/wbguo/apps/anaconda3/lib/python3.8/threading.py", line 932, in _bootstrap_inner
    self.run()
  File "/home/wbguo/apps/anaconda3/lib/python3.8/threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/pool.py", line 541, in _handle_tasks
    cache[job]._set(idx, (False, e))
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/pool.py", line 778, in _set
    self._error_callback(self._value)
  File "../../Simulate/BSReadSim_queue.py", line 274, in process_error
    raise error
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/pool.py", line 537, in _handle_tasks
    put(task)
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/connection.py", line 206, in send
    self._send_bytes(_ForkingPickler.dumps(obj))
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/reduction.py

KeyboardInterrupt: 

Process ForkPoolWorker-171:
Process ForkPoolWorker-169:
Process ForkPoolWorker-168:
Process ForkPoolWorker-172:
Process ForkPoolWorker-170:
Process ForkPoolWorker-166:
Process ForkPoolWorker-164:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/home/wbguo/apps/anaconda

  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/connection.py", line 216, in recv_bytes
    buf = self._recv_bytes(maxlength)
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
  File "/home/wbguo/apps/anaconda3/lib/python3.8/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
Traceback (

In [25]:
seed = None

In [26]:
if seed and seed < 0:
    print(seed)

In [24]:
seed

In [12]:
self.processed_queue.qsize()

0

In [13]:
pool.close()

In [14]:
pool.join()

In [15]:
self.fastq_out.data_queue.qsize()

0

In [ ]:
import queue
import threading
import multiprocessing

data_queue= queue.Queue()
writer_thread = threading.Thread(target=self.fastq_out.output_queue, args=(data_queue,))
writer_thread.start()

for contig_id in self.count_dict.keys():
    if self.count_dict[contig_id] == 0: # need to test if reads < self.n_threads TODO:
        continue
    sim_cmd  = self.cmd_part + ['-c', contig_id] + ['-n', str(self.count_dict[contig_id])]
    read_gen = LockedIterator(StreamHTSIM(sim_cmd=sim_cmd, pair_end=self.pair_end)) # only output 1 header for -c TODO:
    var_contig, sim_data= next(read_gen)                                    # the first element of generator is variants
    self.current_contig = var_contig                                        # update the profiles
    self.pos_map, self.meth_arr, _ = self.meth_db.load_contig(var_contig)   # [pos_map, meth_arr, status]
    self.variant_profile= self.meth_set.set_var_meth(var_contig, sim_data)  # a dict, can be empty

    def write_data(data):
        data_queue.put(data)

    pool = multiprocessing.Pool(processes=4)
    with tqdm(total=self.count_dict[contig_id]) as progress:
        for _, read_pair in read_gen:
            pool.apply_async(self.process_read_pair, args=(read_pair,), callback=write_data)
            progress.update(1)

     # Wait for all processes to complete
    pool.close()
    pool.join()

data_queue.put(None)
writer_thread.join()
self.fastq_out.close()

In [ ]:
data_queue.get()

# var_profile and generator

In [ ]:
contig_id = 'chr10'
sim_cmd  = self.cmd_part + ['-c', contig_id] + ['-n', str(self.count_dict[contig_id])]

In [ ]:
' '.join(sim_cmd)

In [ ]:
read_gen = LockedIterator(StreamHTSIM(sim_cmd=sim_cmd, pair_end=self.pair_end))
var_contig, sim_data= next(read_gen)
self.current_contig = var_contig
self.pos_map, self.meth_arr, _ = self.meth_db.load_contig(var_contig)
self.variant_profile= self.meth_set.set_var_meth(var_contig, sim_data)

In [ ]:
var_contig

In [ ]:
len(self.pos_map)

In [ ]:
sim_data # 0-based

In [ ]:
self.variant_profile

In [ ]:
for _, read_pair in read_gen:
    if len(read_pair[0]['seq']) != len(read_pair[0]['ctx']):
        break
    if len(read_pair[1]['seq']) != len(read_pair[1]['ctx']):
        break
    if read_pair[0]['n_sub'] !=0:
        break

# check whole process

In [ ]:
read_pair

In [ ]:
print(f"Read1:{''.join(['ACGT'[i] for i in read_pair[0]['seq']])}\nRead2:{''.join(['ACGT'[i] for i in read_pair[1]['seq']])}")

In [ ]:
read1_idx   = random.choice([0, 1]) if read_pair[0]['strand']<0 else read_pair[0]['strand']
pattern_idx = random.choice([0, 1]) if self.undirectional else read1_idx

self.mask_context(read_pair[0], pattern_idx)
self.mask_context(read_pair[1], pattern_idx)

# retrive methy profile
self.retrive_meth_db(read_pair[0])
self.retrive_meth_db(read_pair[1])

# set methylation states
self.set_context_state(read_pair)

# bisulfite converted
self.treat_bisulfite(read_pair[0])
self.treat_bisulfite(read_pair[1])

# rev complementary
self.rev_complement(read_pair)

# introduce seq errors
self.add_seq_err(read_pair[read1_idx], pattern_idx)
self.add_seq_err(read_pair[1-read1_idx],1-pattern_idx)

# introduce quality scores
self.add_qual_score(read_pair[0])
self.add_qual_score(read_pair[1])

self.fastq_out.output_reads(read_pair, read1_idx, pattern_idx)

In [ ]:
str(self.ref_dict['chr10'][131712:131812].seq.reverse_complement())

# mask

In [ ]:
read_pair

In [ ]:
read1_idx   = random.choice([0, 1]) if read_pair[0]['strand']<0 else read_pair[0]['strand']
pattern_idx = random.choice([0, 1]) if self.undirectional else read1_idx

In [ ]:
[read1_idx, pattern_idx]

In [ ]:
self.mask_context(read_pair[0], pattern_idx)
self.mask_context(read_pair[1], pattern_idx)

In [ ]:
read_pair

# retrive

In [ ]:
self.retrive_meth_db(read_pair[0])
self.retrive_meth_db(read_pair[1])

In [ ]:
read_pair[1]

In [ ]:
read_rec = read_pair[1]
read_meth = np.zeros(self.read_len)
site_flag = np.logical_not(read_rec['ctx'].mask)            # unmasked sites

if np.any(site_flag):                                       # contain methylable bases
    arr_idx  = 2
    read_pos = read_rec['start'] + np.arange(self.read_len)

    if read_rec['flag_pos']:                                # covers mutation position
        if self.asm_sim:
            arr_idx  = 4 if read_rec['flag_mut'] else 3

        if read_rec['n_indel']:                             # handle indel first (offset)
            read_pos += read_rec['ofs']
            indel_site= read_rec['cgr'] == 3
            read_meth[indel_site]= self.fetch_meth_val(read_pos[indel_site], arr_idx, 3)

        if read_rec['n_sub']:
            snp_site = site_flag & (read_rec['cgr'] == 1)   # snp methylable site
            read_meth[snp_site]  = self.fetch_meth_val(read_pos[snp_site],  arr_idx, 1)

        match_site = site_flag & (read_rec['cgr'] == 0)     # match methylable site
        read_meth[match_site] = self.fetch_meth_val(read_pos[match_site],arr_idx, 0)
    else:
        match_site = site_flag                              # SNP/INDEL free region
        read_meth[match_site] = self.fetch_meth_val(read_pos[match_site],arr_idx, 0)
read_rec['meth']   = read_meth
read_rec['pos']    = read_pos

In [ ]:
read_pair

In [ ]:
read_pos

In [ ]:
self.variant_profile

# set context state

In [ ]:
self.set_context_state(read_pair)

In [ ]:
read_pair

# treat bisulfite

In [ ]:
self.treat_bisulfite(read_pair[0])
self.treat_bisulfite(read_pair[1])

In [ ]:
read_rec = read_pair[1]

In [ ]:
unmeth_idx = np.where(np.bitwise_and(read_rec['ctx'], 0x1)==1)[0] # behave strange without ==1

In [ ]:
np.where(np.bitwise_and(read_rec['ctx'], 0x1)==1)[0]

In [ ]:
read_rec['meth'].size

In [ ]:
unmeth_idx

In [ ]:
conv_states= bernoulli.rvs(self.conversion_rate, size=unmeth_idx.size)

In [ ]:
conv_states==1

In [ ]:
unmeth_idx

In [ ]:
unmeth_idx[conv_states==1]

In [ ]:
read_pair

In [ ]:
print(f"Read1:{''.join(['ACGT'[i] for i in read_pair[0]['seq']])}\nRead2:{''.join(['ACGT'[i] for i in read_pair[1]['seq']])}")

# reverse complement

In [ ]:
self.rev_complement(read_pair)

In [ ]:
read_pair

In [ ]:
print(f"Read1:{''.join(['ACGT'[i] for i in read_pair[0]['seq']])}\nRead2:{''.join(['ACGT'[i] for i in read_pair[1]['seq']])}")

# add seq err

In [ ]:
self.add_seq_err(read_pair[read1_idx], pattern_idx)
self.add_seq_err(read_pair[1-read1_idx],1-pattern_idx)

In [ ]:
read_rec = read_pair[read1_idx]
pattern_idx = pattern_idx

for i in range(100000):
    err_idx = np.where(bernoulli.rvs(self.err_rate, size = self.read_len))[0] #cannot np.squeeze
    if np.any(err_idx):
        for idx in err_idx:
            base_ori = read_rec['seq'][idx]
            base_err = np.random.choice(np.setdiff1d(np.array([0,1,2,3]), base_ori), size = 1)[0]
            read_rec['seq'][idx] = base_err
            read_rec['cgr'][idx] = 2
            read_rec['ctx'][idx] = 5 if (base_ori, base_err)==[(1,3), (2,0)][pattern_idx] else 6

In [ ]:
read_pair

In [ ]:
err_idx =np.where(bernoulli.rvs(self.err_rate, size = self.read_len))

In [ ]:
err_idx

In [ ]:
read_rec = read_pair[1]
for idx in err_idx:
    base_ori = read_rec['seq'][idx]

In [ ]:
base_ori

In [ ]:
base_err = np.random.choice(np.setdiff1d(np.array([0,1,2,3]), base_ori), size = 1)

In [ ]:
base_err

In [ ]:
print(f"Read1:{''.join(['ACGT'[i] for i in read_pair[0]['seq']])}\nRead2:{''.join(['ACGT'[i] for i in read_pair[1]['seq']])}")

# add qual score

In [ ]:
self.add_qual_score(read_pair[0])
self.add_qual_score(read_pair[1])

In [ ]:
read_pair

# output

In [ ]:
self.fastq_out.output_reads(read_pair, read1_idx, pattern_idx)

In [ ]:
read_rec = read_pair[1]
for ix, ctx in enumerate(read_rec["ctx"]):
    print(ctx)

In [ ]:
read_rec["ctx"]

In [ ]:
read_rec["cgr"]

# save pickle

In [ ]:
# import pickle

# with open('/home/wbguo/iproject/BSReadSim/test/data/test_read_pair.pickle', 'wb') as handle:
#     pickle.dump(read_pair, handle)

# get reference

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq

In [ ]:
str(self.ref_dict['chr10'][18901:19001].seq)

In [ ]:
x = Seq("GAAATACAGATTCCTCGGCACCACCCGAGACCTACTGAATCAGACACAGTAGTAAAAATAAAGATAGTAGGGGCCAGGCGCGGTGGCTCATACCTGTAAC")

In [ ]:
str(x.reverse_complement())